# ターゲティング② — 改善モデル：勾配ブースティングと施策の深さをテストで測る

出典：第03章 04節／パス：`03_targeting/02_improved.ipynb`／パッケージ：numpy・pandas・scikit-learn・scipy／実行時間の目安：約2分

このノート単体で実行できる。データ取得と分け方は①と同一であり、失敗時は `input/bank.csv`（合成代替）、さらに欠品のときは内蔵生成へ切り替わる。
ベースラインのコードは `01_baseline.ipynb` に分けてある。施策の深さは検証用で選び、最終評価はテストセットで1回だけ行う。
接触原価・成約粗利の数値は仮置きであり、実務の根拠ではない。


In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RS = 0
BANK_URL = "https://openml.org/data/v1/download/1586218/bank-marketing.arff"
# OpenML版の列名 V1〜V16 を UCI Bank Marketing の列名に戻す（列の順はUCIと同じ）
COLS = ["age", "job", "marital", "education", "default", "balance", "housing", "loan",
        "contact", "day", "month", "duration", "campaign", "pdays", "previous", "poutcome"]
# 通話時間は通話が終わるまで決まらない。架電先を選ぶ時点には存在しない（リーク列）
LEAK = ["duration"]


def make_bank_data(n: int, seed: int = 3) -> pd.DataFrame:
    # 公開データと同じ列名・型の合成代替（取得失敗時のみ使う手順確認用）
    r = np.random.default_rng(seed)
    prev = (r.poisson(1.5, size=n) + 1) * (r.uniform(size=n) < 0.18)
    d = pd.DataFrame({
        "age": r.integers(18, 90, size=n).astype(float),
        "job": r.choice(["blue-collar", "management", "technician", "admin.", "services", "retired"], size=n),
        "marital": r.choice(["married", "single", "divorced"], size=n, p=[0.60, 0.28, 0.12]),
        "education": r.choice(["secondary", "tertiary", "primary", "unknown"], size=n, p=[0.51, 0.29, 0.16, 0.04]),
        "default": r.choice(["no", "yes"], size=n, p=[0.98, 0.02]),
        "balance": np.round(r.lognormal(6.0, 1.5, size=n) - 300.0),
        "housing": r.choice(["yes", "no"], size=n, p=[0.56, 0.44]),
        "loan": r.choice(["no", "yes"], size=n, p=[0.84, 0.16]),
        "contact": r.choice(["cellular", "unknown", "telephone"], size=n, p=[0.65, 0.29, 0.06]),
        "day": r.integers(1, 32, size=n).astype(float),
        "month": r.choice(["may", "jul", "aug", "jun", "nov", "apr"], size=n),
        "duration": np.round(r.exponential(250.0, size=n)),
        "campaign": (r.poisson(1.8, size=n) + 1).astype(float),
        "pdays": np.where(prev > 0, r.integers(1, 400, size=n), -1).astype(float),
        "previous": prev.astype(float),
        "poutcome": np.where(prev > 0, r.choice(["failure", "other", "success"], size=n, p=[0.6, 0.15, 0.25]), "unknown"),
    })
    logit = (-3.6 + 0.004 * d["duration"] - 0.15 * d["campaign"] + 1.8 * (d["poutcome"] == "success")
             + 0.5 * (d["housing"] == "no") + 0.4 * (d["contact"] == "cellular"))
    y = (r.uniform(size=n) < 1.0 / (1.0 + np.exp(-logit))).astype(int)
    d["Class"] = np.where(y == 1, 2, 1)
    return d


def load_bank() -> pd.DataFrame:
    import io
    import urllib.request
    from scipy.io import arff
    raw = urllib.request.urlopen(BANK_URL, timeout=60).read()
    data, _ = arff.loadarff(io.StringIO(raw.decode("utf-8")))
    return pd.DataFrame(data)


def split3(X, y):
    # テスト25%を先に切り出し、残りを学習75%・検証25%に分ける（①②のノートで共通）
    X_tmp, X_te, y_tmp, y_te = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RS)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=RS)
    return X_tr, X_va, X_te, y_tr, y_va, y_te


def make_pre(X):
    num = [c for c in X.columns if X[c].dtype != object]
    cat = [c for c in X.columns if X[c].dtype == object]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])


try:
    raw = load_bank()
    for c in raw.columns:
        if raw[c].dtype == object:
            raw[c] = raw[c].str.decode("utf-8")
    print("公開データを使用: OpenML bank-marketing（銀行販促の実データ）")
except Exception:
    try:
        raw = pd.read_csv("input/bank.csv")
        print("input/bank.csv（合成代替）を使用")
    except FileNotFoundError:
        raw = make_bank_data(45211)
        print("合成データを生成（手順確認用）")
raw = raw.rename(columns=dict(zip([f"V{j}" for j in range(1, 17)], COLS)))
y = (raw["Class"].astype(str) == "2").astype(int)  # 購読あり
X_all = raw.drop(columns=["Class"])
X = X_all.drop(columns=LEAK)
num_cols = [c for c in X.columns if X[c].dtype != object]
cat_cols = [c for c in X.columns if X[c].dtype == object]
print(f"件数: {len(raw)}, 購読率: {y.mean():.3f} / 数値{len(num_cols)}列・カテゴリ{len(cat_cols)}列（{', '.join(LEAK)}を除く）")


公開データを使用: OpenML bank-marketing（銀行販促の実データ）
件数: 45211, 購読率: 0.117 / 数値6列・カテゴリ9列（durationを除く）


In [2]:
X_tr, X_va, X_te, y_tr, y_va, y_te = split3(X, y)
hgb = make_pipeline(
    make_pre(X),
    HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05,
                                   max_leaf_nodes=15, min_samples_leaf=50, random_state=RS),
).fit(X_tr, y_tr)
p_va = hgb.predict_proba(X_va)[:, 1]
p_te = hgb.predict_proba(X_te)[:, 1]
print(f"[改善モデル・HGB] テストPR-AUC={average_precision_score(y_te, p_te):.4f}")
print(f"学習: {len(X_tr)}件 / 検証: {len(X_va)}件 / テスト: {len(X_te)}件")

[改善モデル・HGB] テストPR-AUC=0.4453
学習: 25431件 / 検証: 8477件 / テスト: 11303件


In [3]:
# 施策の深さ：反応確率の上位k割に架電したときの期待利益（費用・粗利は仮置き）
# 架電1件の接触原価1,000円、成約1件の粗利6,000円と仮定する
# 期待利益 = 成約1件の粗利 × 成約（反応者）数 − 接触原価 × 架電数
COST_CALL, GAIN = 1000.0, 6000.0
FRACS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]


def profit_at(p, y_true, frac):
    ys = y_true.to_numpy()[np.argsort(-p)]
    k = int(len(ys) * frac)
    hit = int(ys[:k].sum())
    return k, hit, GAIN * hit - COST_CALL * k


print("上位層 / 架電数 / 成約数 / 期待利益(円)［検証用］:")
rows = []
for frac in FRACS:
    k, hit, prof = profit_at(p_va, y_va, frac)
    rows.append((frac, prof))
    print(f"{frac:.2f} / {k:4d} / {hit:4d} / {prof:,.0f}")
best = max(rows, key=lambda t: t[1])
near = [f for f, pr in rows if pr >= 0.9 * best[1]]
print(f"最大利益の深さ: 上位{best[0]:.2f}（{best[1]:,.0f}円）［検証用で選択］")
print(f"最大の9割以上を保つ深さ: 上位{min(near):.2f}〜{max(near):.2f}（この幅では利益がほぼ平ら）")
# 最終評価はテストセットで1回だけ
k, hit, profit_te = profit_at(p_te, y_te, best[0])
print(f"[テスト] 上位{best[0]:.2f} / 架電数 {k} / 成約数 {hit} / 期待利益 {profit_te:,.0f}円")
assert np.isfinite(p_te).all()

上位層 / 架電数 / 成約数 / 期待利益(円)［検証用］:
0.05 /  423 /  268 / 1,185,000
0.10 /  847 /  443 / 1,811,000
0.15 / 1271 /  561 / 2,095,000
0.20 / 1695 /  633 / 2,103,000
0.25 / 2119 /  685 / 1,991,000
0.30 / 2543 /  721 / 1,783,000
0.40 / 3390 /  778 / 1,278,000
0.50 / 4238 /  824 / 706,000
最大利益の深さ: 上位0.20（2,103,000円）［検証用で選択］
最大の9割以上を保つ深さ: 上位0.15〜0.25（この幅では利益がほぼ平ら）
[テスト] 上位0.20 / 架電数 2260 / 成約数 819 / 期待利益 2,654,000円


In [4]:
# 改善結果：ベースラインと改善モデルの差をテストで1回だけ測る
# ベースラインは01_baselineと同じ分け方・同じ学習データ（学習用）で作り直す
lr0 = make_pipeline(make_pre(X), LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
p0 = lr0.predict_proba(X_te)[:, 1]
a0 = average_precision_score(y_te, p0)
a1 = average_precision_score(y_te, p_te)
verdict = "ほぼ並んだため説明しやすい線形を本番に推す" if (a1 - a0) < 0.02 else "改善幅があるためHGBを本番に推す"
print(f"[改善結果] PR-AUC {a0:.4f} → {a1:.4f}（差 {a1 - a0:+.4f}。{verdict}）")
_, _, profit0 = profit_at(p0, y_te, best[0])
print(f"[改善結果] 上位{best[0]:.2f}に架電したときのテスト期待利益 {profit0:,.0f}円 → {profit_te:,.0f}円")

[改善結果] PR-AUC 0.4137 → 0.4453（差 +0.0315。改善幅があるためHGBを本番に推す）
[改善結果] 上位0.20に架電したときのテスト期待利益 2,228,000円 → 2,654,000円


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
profs = [profit_at(p_va, y_va, f)[2] for f in FRACS]
plt.figure(); plt.bar([str(f) for f in FRACS], profs)
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("top fraction"); plt.ylabel("expected profit (JPY)"); plt.title("profit by depth (validation)")
plt.savefig(f"{FIG}/code-result-j03-02.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j03-02.png")

saved code-result-j03-02.png


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
ys0 = y_te.to_numpy()[np.argsort(-p0)]
ys1 = y_te.to_numpy()[np.argsort(-p_te)]
fracs = [0.05, 0.10, 0.20, 0.30]
base = ys0.mean()
l0 = [ys0[:int(len(ys0) * f)].mean() / base for f in fracs]
l1 = [ys1[:int(len(ys1) * f)].mean() / base for f in fracs]
x = np.arange(len(fracs)); w = 0.35
plt.figure()
plt.bar(x - w / 2, l0, w, label="baseline")
plt.bar(x + w / 2, l1, w, label="improved")
plt.xticks(x, [str(f) for f in fracs])
plt.xlabel("top fraction"); plt.ylabel("lift"); plt.legend()
plt.title("lift before vs after (test)")
plt.savefig(f"{FIG}/code-result-j03-03.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j03-03.png")

saved code-result-j03-03.png


## 読み取りどころ

- 深く架電するほど成約は拾えるが、接触原価で利益が削られるトレードオフ。
- 最大利益の深さが全件ではないこと（ばらまきは損）。最大付近では利益が平らで、深さの選択は幅を持って読むこと。
- 予測精度と施策効果は別——モデルが当てるのは「架電したら成約するか」であり、架電しなくても成約した人との区別（アップリフト）は別に検証が要ること。
